il modello della lezione è un ottimo punto di partenza, ma nel mondo reale dobbiamo bilanciare precisoine e velovità. Sfida te stesso con queste modifiche:
- Riduzione carico: modifica il caricamento dati per usare solo le prime 5_000 parole e riduci il padding (maxlen) a 100
- Architettura efficiente: riduci la dimensione dell'embedding a 32 e usa una LSTM puù snella da 32 unità
- Regolarizzazione: aggiungi un layer dropout esplicito del 30% tra la LSTM e il layer Dense finale per combattere l'overfitting
- Analisi: addestra per 5 epoche e osserva se l'accuratezza rimane stabile nonostande i parametri ridotti.


In [1]:
import keras
from keras import layers, models
from keras.datasets import imdb
from keras.preprocessing.sequence import pad_sequences

# --- 1. CONFIGURAZIONE E CARICAMENTO DATI ---
# MODIFICA 1: Riduzione vocabolario a 5.000 parole
max_features = 5000 
# MODIFICA 2: Riduzione lunghezza sequenza a 100
maxlen = 100 

print("Caricamento dati...")
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

# --- 2. PREPROCESSING: PADDING ---
x_train = pad_sequences(x_train, maxlen=maxlen, padding='pre')
x_test = pad_sequences(x_test, maxlen=maxlen, padding='pre')

# --- 3. DEFINIZIONE DELL'ARCHITETTURA ---
# 
model = models.Sequential([
    # MODIFICA 3: Embedding ridotto a 32 dimensioni (più veloce da addestrare)
    layers.Embedding(input_dim=max_features, output_dim=32, name="Semantics_Space"),
    
    # MODIFICA 4: LSTM snella da 32 unità
    layers.LSTM(32, dropout=0.2, recurrent_dropout=0.2, name="Context_Processor"),
    
    # MODIFICA 5: Aggiunta layer Dropout esplicito al 30% per ridurre l'overfitting
    layers.Dropout(0.3),
    
    layers.Dense(1, activation='sigmoid', name="Sentiment_Classifier")
])

# --- 4. COMPILAZIONE E TRAINING ---
model.compile(optimizer='adam', 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

print("Inizio addestramento...")
# MODIFICA 6: Incremento a 5 epoche per osservare la stabilità
history = model.fit(x_train, y_train, 
                    epochs=5, 
                    batch_size=32, 
                    validation_split=0.2)

# Valutazione finale
results = model.evaluate(x_test, y_test)
print(f"Test Loss: {results[0]:.4f}, Test Accuracy: {results[1]:.4f}")

# --- 5. FUNZIONE DI PREDIZIONE (Invariata ma adattata ai nuovi limiti) ---
word_index = imdb.get_word_index()

def predict_sentiment(text):
    words = text.lower().split()
    tokenized = []
    for word in words:
        index = word_index.get(word, -3)
        actual_index = index + 3
        if actual_index < max_features:
            tokenized.append(actual_index)
        else:
            tokenized.append(2)
            
    padded_text = pad_sequences([tokenized], maxlen=maxlen)
    prediction = model.predict(padded_text, verbose=0)[0][0]
    
    sentiment = "POSITIVA" if prediction > 0.5 else "NEGATIVA"
    print(f"\nRecensione: \"{text}\" -> {sentiment} ({prediction*100:.2f}%)")

# Test rapidi
predict_sentiment("this movie was great")
predict_sentiment("it was a total disaster")

Caricamento dati...
Inizio addestramento...
Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 62s 85ms/step - accuracy: 0.7374 - loss: 0.5203 - val_accuracy: 0.8142 - val_loss: 0.4067
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 98s 111ms/step - accuracy: 0.8402 - loss: 0.3782 - val_accuracy: 0.8270 - val_loss: 0.3899
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 81s 129ms/step - accuracy: 0.8631 - loss: 0.3307 - val_accuracy: 0.8278 - val_loss: 0.3831
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 72s 115ms/step - accuracy: 0.8781 - loss: 0.2986 - val_accuracy: 0.8330 - val_loss: 0.3797
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 52s 82ms/step - accuracy: 0.8921 - loss: 0.2724 - val_accuracy: 0.8348 - val_loss: 0.3931
782/782 ━━━━━━━━━━━━━━━━━━━━ 17s 22ms/step - accuracy: 0.8408 - loss: 0.3867
Test Loss: 0.3867, Test Accuracy: 0.8408

Recensione: "this movie was great" -> POSITIVA (75.41%)

Recensione: "it was a total disaster" -> NEGATIVA (27.85%)
